In [2]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

In [3]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')

y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)
print(total_data['target'])

ROW_ID
0         0.009210
1         0.000405
2         0.005967
3        -0.000542
4        -0.002579
            ...   
527068   -0.001769
527069    0.005640
527070    0.004016
527071    0.001617
527072    0.000556
Name: target, Length: 527073, dtype: float64


In [4]:
ret_1_pos=total_data[np.sign(total_data['RET_1'])==np.sign(total_data['target'])]
ret_1_neg=total_data[np.sign(total_data['RET_1'])!=np.sign(total_data['target'])]
for i in range(2,20):
    ret_i_pos=ret_1_pos['RET_'+str(i)]
    ret_i_neg=ret_1_neg['RET_'+str(i)]



In [5]:
data = total_data.copy()
data['actual'] = (data['target'] > 0).astype(int)
data['ret1_pred'] = (data['RET_1'] > 0).astype(int)
data['ret1_correct'] = data['ret1_pred'] == data['actual']
data['ret1_failure'] = ~data['ret1_correct']

for i in range(2, 21):
    lag = f'RET_{i}'
    data[f'{lag}_same_sign'] = (
        np.sign(data['RET_1']) == np.sign(data[lag])
    )

    print(
        lag,
        data.groupby('ret1_failure', observed=True)[f'{lag}_same_sign']
            .mean()
            .to_dict()
    )

RET_2 {False: 0.5115041076958513, True: 0.5175223416758031}
RET_3 {False: 0.4964224473604352, True: 0.4971801768403783}
RET_4 {False: 0.4961518915144181, True: 0.5014552653788817}
RET_5 {False: 0.4965321321628746, True: 0.497188064457608}
RET_6 {False: 0.5016361316363875, True: 0.4984106451282132}
RET_7 {False: 0.5051972315555864, True: 0.49721961492652683}
RET_8 {False: 0.5083415292255156, True: 0.49974759624864923}
RET_9 {False: 0.5092409446055186, True: 0.5010529969001665}
RET_10 {False: 0.5055591914036364, True: 0.5017273881733068}
RET_11 {False: 0.5044075009780228, True: 0.49884840788446216}
RET_12 {False: 0.5052228246761556, True: 0.49778752336706605}
RET_13 {False: 0.5057493117278647, True: 0.5024372737239807}
RET_14 {False: 0.5047402115454223, True: 0.4999329552535475}
RET_15 {False: 0.503884670086395, True: 0.5017313319819215}
RET_16 {False: 0.5049193633894067, True: 0.5016563996182394}
RET_17 {False: 0.502634263338586, True: 0.5040739542991458}
RET_18 {False: 0.50124126634760

In [6]:
for i in range(2, 21):
    lag = f'RET_{i}'
    data[f'{lag}_agrees_with_ret1'] = (
        np.sign(data['RET_1']) == np.sign(data[lag])
    )

    for condition, mask in {
        'agreement': data[f'{lag}_agrees_with_ret1'],
        'disagreement': ~data[f'{lag}_agrees_with_ret1'],
    }.items():
        accuracy = data.loc[mask, 'ret1_correct'].mean()
        count = mask.sum()
        print(lag, condition, count, accuracy)

RET_2 agreement 271126 0.5160036293088822
RET_2 disagreement 255947 0.5220182303367494
RET_3 agreement 261843 0.5185435547255416
RET_3 disagreement 265230 0.5193002299890661
RET_4 agreement 262853 0.5162695499005148
RET_4 disagreement 264220 0.5215653621981682
RET_5 agreement 261875 0.5185947494033413
RET_5 disagreement 265198 0.5192497680977987
RET_6 agreement 263581 0.5205344846555707
RET_6 disagreement 263492 0.5173136186297876
RET_7 agreement 264253 0.5228966180137974
RET_7 disagreement 262820 0.514930370595845
RET_8 agreement 265754 0.5231793312612416
RET_8 disagreement 261319 0.5145971016267474
RET_9 agreement 266331 0.522969537905839
RET_9 disagreement 260742 0.5147924001503402
RET_10 agreement 265495 0.5208233676717076
RET_10 disagreement 261578 0.5169968422420845
RET_11 agreement 264450 0.5216903006239365
RET_11 disagreement 262623 0.5161391043434885
RET_12 agreement 264404 0.5226244686162085
RET_12 disagreement 262669 0.5151997380733927
RET_13 agreement 265727 0.5205643385880

In [7]:
data['RET_1_abs'] = data['RET_1'].abs()
data['RET_1_bin'] = pd.qcut(data['RET_1_abs'], 10, duplicates='drop')

print(
    data.groupby(['ret1_failure', 'RET_1_bin'], observed=True)
        .size()
        .unstack(fill_value=0)
)

RET_1_bin     (-0.00099999822, 0.000268]  (0.000268, 0.000544]  \
ret1_failure                                                     
False                              26435                 26573   
True                               26273                 26134   

RET_1_bin     (0.000544, 0.000839]  (0.000839, 0.00116]  (0.00116, 0.00152]  \
ret1_failure                                                                  
False                        26906                26792               27150   
True                         25801                25915               25558   

RET_1_bin     (0.00152, 0.00195]  (0.00195, 0.00249]  (0.00249, 0.00325]  \
ret1_failure                                                               
False                      27452               27488               27864   
True                       25255               25219               24843   

RET_1_bin     (0.00325, 0.00461]  (0.00461, 0.129]  
ret1_failure                                        
False  

In [9]:
data['RET_1_relative'] = (
    data['RET_1']
    - data.groupby('TS')['RET_1'].transform('mean')
)

data['RET_1_rank'] = (
    data.groupby('TS')['RET_1'].rank(pct=True)
)
data['relative_bin'] = pd.qcut(
    data['RET_1_relative'], 10, duplicates='drop'
)

print(
    data.groupby('relative_bin', observed=True)['ret1_correct']
        .agg(['mean', 'count'])
)

                           mean  count
relative_bin                          
(-0.0387, -0.00322]    0.527206  52708
(-0.00322, -0.00195]   0.513480  52707
(-0.00195, -0.00117]   0.510786  52707
(-0.00117, -0.000552]  0.509667  52707
(-0.000552, 3.2e-06]   0.510587  52708
(3.2e-06, 0.000561]    0.504563  52707
(0.000561, 0.00119]    0.514125  52707
(0.00119, 0.00197]     0.521790  52707
(0.00197, 0.00323]     0.532870  52707
(0.00323, 0.128]       0.544168  52708


In [17]:
data['RET_1_RELATIVE_ABS'] = data['RET_1_relative'].abs()
thresholds=np.arange(0.0001, 0.01, 0.0001)
for threshold in thresholds:
    pred = np.where(
        data['RET_1_RELATIVE_ABS'] > threshold,
        (data['RET_1'] > 0).astype(int),
        0  # or the majority-class prediction
    )
    data['relative_abs_bin'] = pd.qcut(
        data['RET_1_relative'].abs(),
        10,
        duplicates='drop'
    )

    print(
        data.groupby('relative_abs_bin', observed=True)['ret1_correct']
            .agg(['mean', 'count'])
    )

                                 mean  count
relative_abs_bin                            
(-0.000999999578, 0.000275]  0.507949  52708
(0.000275, 0.000557]         0.507200  52707
(0.000557, 0.000855]         0.511811  52707
(0.000855, 0.00118]          0.512057  52707
(0.00118, 0.00154]           0.515045  52708
(0.00154, 0.00196]           0.516990  52707
(0.00196, 0.00249]           0.520405  52707
(0.00249, 0.00322]           0.526477  52707
(0.00322, 0.0045]            0.528070  52707
(0.0045, 0.128]              0.543238  52708
                                 mean  count
relative_abs_bin                            
(-0.000999999578, 0.000275]  0.507949  52708
(0.000275, 0.000557]         0.507200  52707
(0.000557, 0.000855]         0.511811  52707
(0.000855, 0.00118]          0.512057  52707
(0.00118, 0.00154]           0.515045  52708
(0.00154, 0.00196]           0.516990  52707
(0.00196, 0.00249]           0.520405  52707
(0.00249, 0.00322]           0.526477  52707
(0.00322, 

In [22]:
data['fallback_pred'] = 1
max_score = 0
best_threshold = None
for threshold in thresholds:
    data['rule_pred'] = np.where(
        data['RET_1_RELATIVE_ABS'] >= threshold,
        (data['RET_1'] > 0).astype(int),
        data['fallback_pred']
    )

    score = accuracy_score(data['actual'], data['rule_pred'])
    if score > max_score:
        max_score = score
        best_threshold = threshold
data['rule_pred'] = np.where(
    data['RET_1_RELATIVE_ABS'] >= threshold,
    (data['RET_1'] > 0).astype(int),
    data['fallback_pred']
)

print(f"Best threshold: {best_threshold}, Best score: {max_score}")

Best threshold: 0.0001, Best score: 0.5186606029904776
